# Leer diccionarios y juntar dataframes

In [14]:
import pandas as pd
import glob
import os

In [15]:
# Ruta relativa desde notebooks/
ruta_diccionarios = "../data/external/terminos/Diccionarios_de_literatura/"

# Obtener todos los archivos Excel
archivos = glob.glob(os.path.join(ruta_diccionarios, "*.xlsx"))
# 
# Lista para almacenar dataframes
dfs = []

In [16]:
archivos

['../data/external/terminos/Diccionarios_de_literatura\\Literatura latinoamericana reporte_P_v1.xlsx',
 '../data/external/terminos/Diccionarios_de_literatura\\Literatura_Colombiana_P_v1.xlsx',
 '../data/external/terminos/Diccionarios_de_literatura\\Literatura_Infantil_P_v1.xlsx',
 '../data/external/terminos/Diccionarios_de_literatura\\Literatura_juvenil_P_v1.xlsx',
 '../data/external/terminos/Diccionarios_de_literatura\\Literatura_universal_reporte_P_v2.xlsx']

In [17]:
for archivo in archivos:
    # Leer archivo
    df_temp = pd.read_excel(archivo)
    
    # Extraer nombre del archivo sin ruta ni extensión
    nombre_archivo = os.path.splitext(os.path.basename(archivo))[0]
    
    # Añadir columna identificadora
    df_temp["fuente"] = nombre_archivo
    
    dfs.append(df_temp)

# Concatenar todo en un solo dataframe
df_total = pd.concat(dfs, ignore_index=True)

print(df_total.shape)

(45415, 10)


In [18]:
df_total.head()

,instance_primary_contributor,contributors,extraido,title,notes,publication_site,publisher,dateOfPublication,pais_origen,fuente
0,Sin valor,Sin valor,True,Hombres y engranajes,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1
1,Sin valor,Sin valor,True,La cautiva ; el matadero,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1
2,Sin valor,Sin valor,True,Homo atomicus,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1
3,Sin valor,Sin valor,True,Obras completas : falsificaciones,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1
4,Sin valor,Sin valor,True,Arengas de Bartolomé Mitre : colección de disc...,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1


In [19]:
df_total["title_author"] = (
    df_total["title"].fillna("").str.strip() +
    " | " +
    df_total["instance_primary_contributor"].fillna("").str.strip()
)


In [20]:
df_total.nunique()

instance_primary_contributor    13128
contributors                    14231
extraido                            2
title                           30883
notes                           12094
publication_site                  691
publisher                        5161
dateOfPublication                 629
pais_origen                        18
fuente                              5
title_author                    31689
dtype: int64

In [21]:
df_total['title'].value_counts()

title
Magazin dominical                                                                                     853
Espigas eucarísticas                                                                                  481
Boletin mensual de estadistica                                                                        446
La defensa católica : periódico doctrinal órgano del consejo superior del apostolado de la oración    226
La buena prensa : periódico semanal, dedicado a defender los intereses religiosos y de la patria      203
                                                                                                     ... 
Mil y una susana silvestre                                                                              1
Cuentos Reunidos 1: Roberto Fontanarrosa                                                                1
Juego de amor                                                                                           1
El camino de ida Ricardo Piglia         

Verificar cuantos nombres hay en común entre instance_primary_contributor y contributors

In [22]:
set_primary = set(df_total["instance_primary_contributor"].dropna())
set_contributors = set(df_total["contributors"].dropna())

no_estan = set_primary - set_contributors

len(no_estan)

3902

In [23]:
en_comun = set_primary & set_contributors
len(en_comun)


9226

Verificar cuantos de latinoamericana están en universal

In [24]:
df_total["fuente"].unique()


array(['Literatura latinoamericana reporte_P_v1',
       'Literatura_Colombiana_P_v1', 'Literatura_Infantil_P_v1',
       'Literatura_juvenil_P_v1', 'Literatura_universal_reporte_P_v2'],
      dtype=object)

In [25]:
# Filtrar cada subconjunto
latam = df_total[df_total["fuente"].str.contains("latinoamericana", case=False)]
universal = df_total[df_total["fuente"].str.contains("universal", case=False)]

# Crear sets únicos de títulos (sin NaN)
titulos_latam = set(latam["title"].dropna())
titulos_universal = set(universal["title"].dropna())

# Intersección
titulos_repetidos = titulos_latam & titulos_universal

len(titulos_repetidos)


55

In [26]:
# Filtrar cada subconjunto
latam = df_total[df_total["fuente"].str.contains("latinoamericana", case=False)]
universal = df_total[df_total["fuente"].str.contains("universal", case=False)]

# Crear sets únicos de títulos (sin NaN)
titulos_latam = set(latam["title_author"].dropna())
titulos_universal = set(universal["title_author"].dropna())

# Intersección
titulos_repetidos = titulos_latam & titulos_universal

len(titulos_repetidos)

4

In [27]:
titulos_repetidos

{'Antologia poetica | Ruben Dario',
 'La otra literatura latinoamericana | Juan Gustavo Cobo Borda',
 'Nueva historia de la gran literatura iberoamericana | Arturo Torres Rioseco',
 'Obras escogidas | Sin valor'}

# Sin tildes y en minúscula

In [28]:
import unicodedata
import pandas as pd

def normalizar(texto):
    # proteger la ñ/Ñ antes de la descomposición
    texto = texto.replace("ñ", "\x00").replace("Ñ", "\x01")
    
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = unicodedata.normalize("NFC", texto)
    
    # restaurar la ñ/Ñ
    texto = texto.replace("\x00", "ñ").replace("\x01", "Ñ")
    return texto

# Copiar dataframe
df_sin_tildes = df_total.copy()

# Aplicar solo a columnas de texto
columnas_texto = df_sin_tildes.select_dtypes(include=["object", "string"]).columns

df_sin_tildes[columnas_texto] = df_sin_tildes[columnas_texto].applymap(normalizar)



C:\Users\karen\AppData\Local\Temp\ipykernel_25400\2629618008.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_sin_tildes[columnas_texto] = df_sin_tildes[columnas_texto].applymap(normalizar)


In [29]:
df_sin_tildes.head()


,instance_primary_contributor,contributors,extraido,title,notes,publication_site,publisher,dateOfPublication,pais_origen,fuente,title_author
0,Sin valor,Sin valor,True,Hombres y engranajes,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,Hombres y engranajes | Sin valor
1,Sin valor,Sin valor,True,La cautiva ; el matadero,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,La cautiva ; el matadero | Sin valor
2,Sin valor,Sin valor,True,Homo atomicus,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,Homo atomicus | Sin valor
3,Sin valor,Sin valor,True,Obras completas : falsificaciones,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,Obras completas : falsificaciones | Sin valor
4,Sin valor,Sin valor,True,Arengas de Bartolome Mitre : coleccion de disc...,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,Arengas de Bartolome Mitre : coleccion de disc...


In [30]:
df_sin_tildes.nunique()

instance_primary_contributor    13111
contributors                    14196
extraido                            2
title                           30870
notes                           12087
publication_site                  655
publisher                        4985
dateOfPublication                 629
pais_origen                        18
fuente                              5
title_author                    31678
dtype: int64

In [31]:
df_sin_tildes.to_excel("../data/processed/diccionarios_literatura_sin_tildes.xlsx", index=False)

In [32]:
ruta_corpus = "../data/processed/corpus_cleaned.xlsx"

corpus = pd.read_excel(ruta_corpus)

corpus.head()

,Diario,Autor,Fecha,Título,Texto,Vínculo,ID,Texto_limpio
0,El Espectador,Gonzalo Hernández,2018-01-01,Fajardo: para nada tibio,"La Coalición Colombia –Partido Alianza Verde, ...",https://web.archive.org/web/20180102104221/htt...,1,"La Coalición Colombia Partido Alianza Verde, P..."
1,El Espectador,Eduardo Barajas Sandoval,2018-01-01,Macedonia de Norte,Las interpretaciones de la historia sirven com...,https://web.archive.org/web/20180102104221/htt...,2,Las interpretaciones de la historia sirven com...
2,El Espectador,Daniel Emilio Rojas Castro,2018-01-01,El nacionalismo según Vargas Llosa,La semana pasada Mario Vargas Llosa publicó un...,https://web.archive.org/web/20180102104221/htt...,3,La semana pasada Mario Vargas Llosa publicó un...
3,El Espectador,Reinaldo Spitaletta,2018-01-01,"Tiempo sagrado, tiempo profano","Pudiera decirse, sin ser una verdad absoluta, ...",https://web.archive.org/web/20180102104221/htt...,4,"Pudiera decirse, sin ser una verdad absoluta, ..."
4,El Espectador,Aura Lucía Mera,2018-01-01,La rebelión de los bueyes,Lo mejor del encierro de Las Ventas fueron los...,https://web.archive.org/web/20180102104221/htt...,5,Lo mejor del encierro de Las Ventas fueron los...


In [33]:
%pip install --upgrade pip
%pip install pyahocorasick

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [34]:
import pandas as pd
from collections import Counter
import ahocorasick  

In [35]:
# --- 1. Normalización (igual que el original, vectorizada con .map) ---
corpus["texto_limpio_norm"] = (
    corpus["Texto_limpio"].astype(str).map(normalizar).str.lower()
)

def normalizar_serie(serie):
    serie = serie.dropna().astype(str)
    normalizado = serie.map(normalizar).str.lower().str.strip()

    tmp = pd.DataFrame({
        "original": serie,
        "normalizado": normalizado
    })
    tmp = tmp[(tmp["normalizado"] != "") & (tmp["normalizado"] != "sin valor")]

    # si varios originales colapsan al mismo normalizado, se conserva el primero
    tmp = tmp.drop_duplicates(subset="normalizado", keep="first")

    return tmp.set_index("normalizado")["original"]

titulos = normalizar_serie(df_sin_tildes["title"])
autores = pd.concat([
    normalizar_serie(df_sin_tildes["instance_primary_contributor"]),
    normalizar_serie(df_sin_tildes["contributors"])
])
autores = autores[~autores.index.duplicated(keep="first")]

corpus_textos = corpus["texto_limpio_norm"].tolist()


In [36]:
# --- 2. Autómata Aho-Corasick: se construye UNA sola vez con todos los términos ---
terminos_unicos = pd.Index(titulos.index).union(autores.index)

automaton = ahocorasick.Automaton()
for term in terminos_unicos:
    automaton.add_word(term, term)
automaton.make_automaton()


# --- 3. Un único recorrido del corpus, cuenta en cuántos documentos aparece cada término ---
def es_limite(c):
    return c is None or not c.isalnum()

contador = Counter()
for texto in corpus_textos:
    encontrados = set()
    for end_idx, termino in automaton.iter(texto):
        start_idx = end_idx - len(termino) + 1
        char_antes = texto[start_idx - 1] if start_idx > 0 else None
        char_despues = texto[end_idx + 1] if end_idx + 1 < len(texto) else None
        if es_limite(char_antes) and es_limite(char_despues):
            encontrados.add(termino)
    contador.update(encontrados)


In [37]:
# --- 4. Construcción vectorizada de las tablas de frecuencia ---

# --- tablas de frecuencia, ahora con columna adicional ---
freq_titulos = pd.DataFrame({
    "termino": titulos.index,
    "termino_original": titulos.values,
    "tipo": "title",
    "count": [contador.get(t, 0) for t in titulos.index]
})
freq_autores = pd.DataFrame({
    "termino": autores.index,
    "termino_original": autores.values,
    "tipo": "author",
    "count": [contador.get(a, 0) for a in autores.index]
})

tabla_frecuencias = pd.concat([freq_titulos, freq_autores], ignore_index=True)
tabla_frecuencias = (
    tabla_frecuencias[tabla_frecuencias["count"] > 0]
    .sort_values(["tipo", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

In [39]:
tabla_frecuencias.head(50)

,termino,termino_original,tipo,count
0,colombia,Colombia,author,6964
1,medellin,Medellin,author,774
2,españa,España,author,682
3,samper,Samper,author,244
4,german vargas,German Vargas,author,230
5,banco de la republica,Banco de la Republica,author,228
6,oea,Oea,author,172
7,universidad nacional de colombia,Universidad Nacional de Colombia,author,120
8,iglesia catolica,Iglesia Catolica,author,112
9,universidad de los andes,Universidad de los Andes,author,110


In [41]:
ruta_salida = "../data/processed/tabla_frecuencias_literatura.xlsx"
os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)
tabla_frecuencias.to_excel(ruta_salida, index=False)
print("Escrito:", ruta_salida)

Escrito: ../data/processed/tabla_frecuencias_literatura.xlsx
